In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
import os
from pathlib import Path
from typing import Optional

def get_file_path(file_path: str) -> str:
    """
    파일 경로를 절대 경로로 변환하는 함수
    
    Args:
        file_path: 상대 경로 또는 절대 경로
    """
    # 파일 경로 확인 및 절대 경로로 변환
    file_path = Path(file_path)
    if not file_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        file_path = project_root / file_path
    
    if not file_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {file_path}")
    
    return str(file_path)

## pikepdf

In [ ]:
from datetime import datetime
from pathlib import Path
import pikepdf

def _decrypt_to_temp_pdf(src_pdf: Path, password: str) -> Path:
    """pikepdf로 PDF를 열어 '복호화된' PDF를 원본과 동일한 위치에 타임스탬프를 붙인 파일명으로 저장 후 경로 반환"""
    # 원본 PDF와 동일한 디렉토리에 저장
    parent_dir = src_pdf.parent
    stem = src_pdf.stem
    suffix = src_pdf.suffix
    
    # 타임스탬프 생성 (YYYYMMDD_HHMMSS 형식)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    new_filename = f"{stem}_{timestamp}{suffix}"
    decrypted_pdf = parent_dir / new_filename

    with pikepdf.open(str(src_pdf), password=password) as pdf:
        pdf.save(str(decrypted_pdf))  # 저장 시 기본적으로 암호가 제거된 형태로 저장됨(열기 암호 제거 목적)
    return decrypted_pdf

# def _decrypt_to_temp_pdf(src_pdf: Path, password: str) -> Path:
#     """pikepdf로 PDF를 열어 '복호화된' 임시 PDF로 저장 후 경로 반환"""
#     tmp_fd, tmp_path = tempfile.mkstemp(suffix=".pdf")
#     os.close(tmp_fd)
#     tmp_pdf = Path(tmp_path)

#     with pikepdf.open(str(src_pdf), password=password) as pdf:
#         pdf.save(str(tmp_pdf))  # 저장 시 기본적으로 암호가 제거된 형태로 저장됨(열기 암호 제거 목적)
#     return tmp_pdf

## docling

In [ ]:
# Docling Loader - PDF를 마크다운으로 변환

# TESSDATA_PREFIX 환경 변수를 모듈 레벨에서 설정
# docling이 import될 때 tesserocr를 초기화할 수 있으므로 미리 설정
_tessdata_path = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
if not os.environ.get('TESSDATA_PREFIX'):
    os.environ['TESSDATA_PREFIX'] = _tessdata_path

# tesserocr를 직접 import하여 환경 변수를 확인하도록 강제
# docling이 내부적으로 tesserocr.get_languages()를 호출할 때 환경 변수를 읽을 수 있도록
try:
    import tesserocr
    # tesserocr를 초기화하여 환경 변수를 확인하도록 강제
    # get_languages()가 환경 변수를 읽지 못하는 문제를 해결하기 위해
    # PyTessBaseAPI를 사용하여 초기화
    _test_api = tesserocr.PyTessBaseAPI(path=_tessdata_path)
    _test_api.End()
except ImportError:
    # tesserocr가 설치되지 않은 경우 무시 (나중에 오류 처리됨)
    pass
except Exception:
    # 초기화 실패는 무시 (나중에 오류 처리됨)
    pass

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption

# Tesseract OCR과 CPU를 명시적으로 지정하는 설정
# PdfFormatOption의 pipeline_options를 통해 ThreadedPdfPipelineOptions 전달
# ThreadedPdfPipelineOptions의 ocr_options에 TesseractOcrOptions 지정
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    TesseractOcrOptions
)
from docling.datamodel.accelerator_options import AcceleratorOptions

# 암호화된 PDF를 다른 라이브러리로 읽어서 암호를 해제하고 임시 파일로 저장
import tempfile
from typing import List

# docling 버전에 따라 PdfBackendOptions의 위치가 다를 수 있어, 두 경로를 모두 시도합니다.
try:
    from docling.document_converter import PdfBackendOptions
except ImportError:
    from docling.datamodel.base_models import PdfBackendOptions

def table_to_markdown(table: List[List]) -> str:
    """
    표 데이터를 마크다운 테이블 형식으로 변환하는 헬퍼 함수
    
    Args:
        table: 2차원 리스트 형태의 표 데이터
    
    Returns:
        마크다운 테이블 문자열
    """
    if not table or len(table) == 0:
        return ""
    
    # 빈 셀을 빈 문자열로 변환
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).strip()
    
    # 표 데이터 정리
    cleaned_table = [[clean_cell(cell) for cell in row] for row in table]
    
    # 최대 컬럼 수 확인
    max_cols = max(len(row) for row in cleaned_table) if cleaned_table else 0
    
    # 모든 행을 동일한 컬럼 수로 맞춤
    normalized_table = []
    for row in cleaned_table:
        normalized_row = row + [""] * (max_cols - len(row))
        normalized_table.append(normalized_row)
    
    if not normalized_table:
        return ""
    
    markdown_lines = []
    
    # 헤더 행 (첫 번째 행)
    header = normalized_table[0]
    markdown_lines.append("| " + " | ".join(header) + " |")
    
    # 구분선
    markdown_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    
    # 데이터 행들
    for row in normalized_table[1:]:
        markdown_lines.append("| " + " | ".join(row) + " |")
    
    return "\n".join(markdown_lines)


def extract_text_from_pdf_with_docling(pdf_path: str, password: str = None) -> str:
    """
    Docling 라이브러리를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수
    
    Docling은 IBM에서 개발한 문서 변환 라이브러리로, 표, 레이아웃, 구조를 잘 보존하며
    마크다운으로 변환합니다.
    
    Tesseract OCR과 CPU 명시적 지정:
    - OCR 모델: Tesseract (PdfFormatOption의 pipeline_options를 통해 명시적으로 지정)
    - 가속기: CPU (AcceleratorOptions를 통해 명시적으로 지정)
    - 암호화된 PDF: pdfplumber로 텍스트와 표를 추출하여 마크다운으로 변환
    - 암호화되지 않은 PDF: docling으로 직접 마크다운 변환
    
    설정 방법:
    - PdfFormatOption의 pipeline_options 파라미터에 ThreadedPdfPipelineOptions 전달
    - ThreadedPdfPipelineOptions의 ocr_options에 TesseractOcrOptions 지정
    - ThreadedPdfPipelineOptions의 accelerator_options에 AcceleratorOptions(device='cpu') 지정
    - sys.platform 변경 없이 공식 API를 통해 명시적으로 설정
    
    주의사항:
    - TesseractOcrOptions를 사용하려면 tesserocr 라이브러리와 올바른 설정이 필요합니다.
    - Tesseract OCR 설정 오류 시 폴백 없이 오류가 발생하여 프로세스가 중지됩니다.
    - AcceleratorOptions(device='cpu')를 통해 CPU 사용이 명시적으로 지정됩니다.
    - TESSDATA_PREFIX 환경 변수가 올바르게 설정되어 있어야 합니다.
    
    Args:
        pdf_path: PDF 파일 경로 (상대 경로 또는 절대 경로)
        password: 암호화된 PDF의 비밀번호 (선택사항)
    
    Returns:
        마크다운 형식으로 변환된 문자열
    
    Raises:
        FileNotFoundError: PDF 파일을 찾을 수 없을 때
        ImportError: docling 라이브러리가 설치되지 않았을 때
        Exception: PDF를 읽을 수 없을 때
    """
    # TESSDATA_PREFIX 환경 변수 확인 및 설정
    # 모듈 레벨에서 이미 설정되었지만, 함수 내에서도 재확인
    tessdata_path = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
    if not os.environ.get('TESSDATA_PREFIX'):
        os.environ['TESSDATA_PREFIX'] = tessdata_path
    
    # 파일 경로 확인 및 절대 경로로 변환    
    pdf_path = get_file_path(pdf_path)

    try:
                
        # Docling 라이브러리 사용
        # Tesseract OCR과 CPU를 명시적으로 지정하여 사용
        
        # TESSDATA_PREFIX 환경 변수 설정 (tesserocr 사용 시 필요)
        tessdata_path = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
        if not os.environ.get('TESSDATA_PREFIX'):
            os.environ['TESSDATA_PREFIX'] = tessdata_path       
        
        # Tesseract OCR 옵션 생성 (실패 시 오류 발생, 폴백 없음)
        # TesseractOcrOptions를 사용하려면 tesserocr 라이브러리와 올바른 설정이 필요합니다.
        tesseract_ocr_opts = TesseractOcrOptions(
            lang=['eng', 'kor'],  # 영어와 한국어 지원
            bitmap_area_threshold=0.5,
            force_full_page_ocr=False,
            path=tessdata_path  # tessdata 경로 명시
        )

        # ThreadedPdfPipelineOptions 생성 (Tesseract OCR과 CPU 지정)
        threaded_pipeline_opts = ThreadedPdfPipelineOptions(
            ocr_options=tesseract_ocr_opts,
            accelerator_options=AcceleratorOptions(device='cpu')
        )
        
        # PdfFormatOption에 pipeline_options 전달
        pdf_option = PdfFormatOption(
            pipeline_options=threaded_pipeline_opts
        )

        def _convert(target_pdf: str, pwd: Optional[str]) -> str:
            backend_opts = PdfBackendOptions(password=pwd) if pwd else None
            pdf_opt = PdfFormatOption(
                pipeline_options=threaded_pipeline_opts,
                backend_options=backend_opts,
            )

            converter = DocumentConverter(
                allowed_formats=[InputFormat.PDF],
                format_options={InputFormat.PDF: pdf_opt},
            )
            result = converter.convert(target_pdf)
            return result.document.export_to_markdown()

        # 1차: Docling에 password까지 포함해서 바로 시도
        try:
            return _convert(pdf_path, password)
        except Exception as e:
            print(f"❌ Docling으로 PDF 마크다운 변환 오류 발생: {e}")
            msg = str(e).lower()
            print(f"   오류 msg: {msg}")

            # 암호가 있는데 Docling이 유효하지 않다고 하면(페이지 -1 등), pikepdf로 먼저 열어서 재시도
            if password and ("not valid" in msg or "inconsistent number of pages" in msg):
                tmp_pdf = _decrypt_to_temp_pdf(Path(pdf_path), password)
                print(f"   _decrypt_to_temp_pdf 호출")
                print(f"   tmp_pdf: {tmp_pdf}")
                try:
                    return _convert(str(tmp_pdf), None)  # 복호화된 PDF는 password 없이 처리
                finally:
                    try:
                        print(f"--- ")
                    except Exception:
                        pass

            raise
    except ImportError as import_err:
        raise
    except Exception as e:
        raise

In [ ]:
from docling.datamodel.pipeline_options import PdfPipelineOptions
# docling v2 문서 예시 기준 import 경로 :contentReference[oaicite:5]{index=5}
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

# ✅ 표 구조 복원에 유리한 권장 백엔드 (기본값이기도 함) :contentReference[oaicite:5]{index=5}
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend

def extract_text_from_pdf_with_docling2(pdf_path: str, password: str = None) -> str:
    """
    Docling 라이브러리를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수

    [해결 1 적용]
    - 표 구조 추출(do_table_structure=True) 활성화
    - 가능한 경우 TableFormerMode.ACCURATE 사용
    - (권장) bitmap_area_threshold를 낮춰 OCR 트리거 강화
    """
    import os
    from pathlib import Path
    from typing import Optional

    # 파일 경로 확인 및 절대 경로로 변환
    pdf_path = get_file_path(pdf_path)

    try:
        # ---- Docling imports (기존 코드 흐름 유지) ----
        from docling.datamodel.base_models import InputFormat
        from docling.document_converter import DocumentConverter, PdfFormatOption

        from docling.datamodel.pipeline_options import (
            ThreadedPdfPipelineOptions,
            TesseractOcrOptions,
        )
        from docling.datamodel.accelerator_options import AcceleratorOptions

        # docling 버전에 따라 PdfBackendOptions 위치가 다를 수 있어, 두 경로를 모두 시도
        try:
            from docling.document_converter import PdfBackendOptions
        except ImportError:
            from docling.datamodel.base_models import PdfBackendOptions

        # (가능하면) TableFormerMode import
        TableFormerMode = None
        try:
            from docling.datamodel.pipeline_options import TableFormerMode as _TableFormerMode
            TableFormerMode = _TableFormerMode
        except Exception:
            TableFormerMode = None

        # ✅ 핵심: PdfPipelineOptions 사용
        pipeline = PdfPipelineOptions()
        pipeline.do_ocr = False  # ✅ OCR 끔 :contentReference[oaicite:2]{index=2}

        # ✅ 표 구조 추출은 켬 (OCR과 무관)
        pipeline.do_table_structure = True
        pipeline.table_structure_options.do_cell_matching = True

        # ✅ 핵심: “백엔드에서 추출한 텍스트”를 우선 사용
        pipeline.force_backend_text = True  # :contentReference[oaicite:6]{index=6}

        # (선택) 어려운 표면 accurate가 유리한 경우가 많음
        try:
            pipeline.table_structure_options.mode = TableFormerMode.ACCURATE
            print(f"   table_structure_options.mode: {pipeline.table_structure_options.mode}")
        except Exception:
            pass

        pipeline.do_picture_classification = False
        pipeline.do_picture_description = False
        pipeline.accelerator_options = AcceleratorOptions(device="cpu")

        # (선택) 표 구조 모델이 페이지 렌더링 이미지를 쓰는 경우가 있어 켜두는 편이 안정적
        # 이건 OCR이 아니라 "렌더링 이미지 생성"입니다.
        if hasattr(pipeline, "generate_page_images"):
            pipeline.generate_page_images = True
            print(f"   generate_page_images: {pipeline.generate_page_images}")

        def _convert(target_pdf: str, pwd: Optional[str]) -> str:
            backend_opts = PdfBackendOptions(password=pwd) if pwd else None

            pdf_opt = PdfFormatOption(
                pipeline_options=pipeline,
                backend=PyPdfiumDocumentBackend,
                backend_options=backend_opts,
            )

            converter = DocumentConverter(
                allowed_formats=[InputFormat.PDF],
                format_options={InputFormat.PDF: pdf_opt},
            )

            result = converter.convert(target_pdf)

            # “그림 placeholder”를 줄이고 텍스트만 최대한 뽑고 싶으면 strict_text=True도 시도 :contentReference[oaicite:8]{index=8}
            return result.document.export_to_markdown()
            # return result.document.export_to_markdown(strict_text=True)

        # 1차: Docling에 password까지 포함해서 바로 시도
        try:
            return _convert(pdf_path, password)

        except Exception as e:
            print(f"❌ Docling으로 PDF 마크다운 변환 오류 발생: {e}")
            msg = str(e).lower()
            print(f"   오류 msg: {msg}")

            # 암호가 있는데 Docling이 유효하지 않다고 하면(페이지 -1 등), pikepdf로 먼저 열어서 재시도
            if password and ("not valid" in msg or "inconsistent number of pages" in msg):
                tmp_pdf = _decrypt_to_temp_pdf(Path(pdf_path), password)
                print("   _decrypt_to_temp_pdf 호출")
                print(f"   tmp_pdf: {tmp_pdf}")
                try:
                    return _convert(str(tmp_pdf), None)  # 복호화된 PDF는 password 없이 처리
                finally:
                    try:
                        print("---")
                    except Exception:
                        pass

            raise

    except ImportError:
        raise
    except Exception:
        raise


In [ ]:
from typing import Optional
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.accelerator_options import AcceleratorOptions

# ✅ 표 구조 복원에 유리한 권장 백엔드 (기본값이기도 함) :contentReference[oaicite:5]{index=5}
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend

try:
    from docling.document_converter import PdfBackendOptions
except ImportError:
    from docling.datamodel.base_models import PdfBackendOptions

from docling_core.types.doc.document import ContentLayer  # :contentReference[oaicite:3]{index=3}

def extract_text_from_pdf_with_docling3(pdf_path: str, password: str = None) -> str:
    pdf_path = get_file_path(pdf_path)

    def _run(do_cell_matching: bool) -> str:
        pipeline = PdfPipelineOptions(
            do_ocr=False,
            do_table_structure=True,
            do_picture_classification=False,
            do_picture_description=False,
            generate_page_images=True,
            images_scale=2.0,  # 레이아웃/테이블 크롭 품질에 도움될 수 있음 :contentReference[oaicite:6]{index=6}
        )

        # ✅ 표 구조 품질 우선
        pipeline.table_structure_options.mode = TableFormerMode.ACCURATE  # :contentReference[oaicite:7]{index=7}
        pipeline.table_structure_options.do_cell_matching = do_cell_matching  # :contentReference[oaicite:8]{index=8}

        # ✅ 표 구조 목적이면 force_backend_text는 끄는 쪽이 안전
        pipeline.force_backend_text = False  # :contentReference[oaicite:9]{index=9}

        pipeline.accelerator_options = AcceleratorOptions(device="cpu")

        backend_opts = PdfBackendOptions(password=password) if password else None

        converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline,
                    backend=DoclingParseV4DocumentBackend,
                    backend_options=backend_opts,
                )
            },
        )

        result = converter.convert(pdf_path)
        # md = result.document.export_to_markdown()
        md = result.document.export_to_markdown(
            included_content_layers={ContentLayer.BODY},
            page_break_placeholder="\n\n<!-- pagebreak -->\n\n",                 # (선택) 페이지 경계 표시 :contentReference[oaicite:5]{index=5}
        )

        # “테이블이 전혀 안 잡혔는지” 빠른 휴리스틱 (파이프 문자 기반)
        # 필요하면 여기서 result.document에서 TableItem 개수를 세는 방식으로 더 정확히 판단 가능 :contentReference[oaicite:10]{index=10}
        return md

    # 1차: 기본(셀 매칭 True)
    md = _run(do_cell_matching=True)
    if "|---" in md or "| ---" in md:
        print("do_cell_matching=True")
        return md

    # 2차: 셀 매칭 False (borderless/매칭 실패 케이스에 유리) :contentReference[oaicite:11]{index=11}
    md2 = _run(do_cell_matching=False)
    print("do_cell_matching=False")
    return md2


## pdfplumber

In [50]:
def extract_text_from_pdf_with_pdfplumber(pdf_path: str, password: str = None) -> str:
    """
    pdfplumber를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수
    
    pdfplumber는 PDF 파일을 텍스트 데이터로 추출하는 라이브러리로, 표, 이미지, 레이아웃 등을 잘 보존합니다.
    암호화된 PDF와 암호화되지 않은 PDF 모두 처리할 수 있습니다.
    """
    try:
        import pdfplumber
    except ImportError:
        raise ImportError(
            "PDF를 처리하기 위해 pdfplumber가 필요합니다.\n"
            "설치 명령: pip install pdfplumber"
        )
    
    markdown_parts = []
    
    try:
        pdf_path = get_file_path(pdf_path)
        # password가 있으면 암호화된 PDF로 처리, 없으면 암호화되지 않은 PDF로 처리
        pdf_kwargs = {"password": password} if password else {}
        
        with pdfplumber.open(pdf_path, **pdf_kwargs) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_content = []
                
                # 표 추출 (표가 있으면 먼저 표를 추출)
                tables = page.extract_tables()
                if tables:
                    for table_idx, table in enumerate(tables):
                        if table:
                            markdown_table = table_to_markdown(table)
                            if markdown_table:
                                page_content.append(markdown_table)
                                page_content.append("")  # 표 다음에 빈 줄 추가
                
                # 텍스트 추출
                text = page.extract_text()
                if text:
                    page_content.append(text)
                
                if page_content:
                    markdown_parts.append("\n".join(page_content))
        
        return "\n\n".join(markdown_parts) if markdown_parts else ""
        
    except Exception as e:
        # 암호화 관련 오류인지 확인
        error_msg = str(e).lower()
        if 'password' in error_msg or 'encrypted' in error_msg or 'incorrect password' in error_msg:
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        raise

## PDF Load

In [51]:
# text 추출

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127_20260105_182813.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

_password = None
# _password = '345678'

document_text = extract_text_from_pdf_with_docling3(_document_file_path, _password)



2026-01-06 16:27:44,527 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-06 16:27:44,529 - INFO - Going to convert document batch...
2026-01-06 16:27:44,529 - INFO - Initializing pipeline for StandardPdfPipeline with options hash f04c5c2df899fb84580b7a3d67d8ef47
2026-01-06 16:27:44,529 - INFO - Accelerator device: 'cpu'
2026-01-06 16:27:45,271 - INFO - Accelerator device: 'cpu'
2026-01-06 16:27:45,818 - INFO - Processing document 하나생명(액티브)_251127.pdf
2026-01-06 16:27:45,867 - ERROR - Stage preprocess failed for run 1: [json.exception.type_error.302] type must be number, but is array
2026-01-06 16:27:45,902 - ERROR - Stage preprocess failed for run 1: [json.exception.type_error.302] type must be number, but is array
2026-01-06 16:27:45,936 - ERROR - Stage preprocess failed for run 1: [json.exception.type_error.302] type must be number, but is array
2026-01-06 16:27:45,969 - ERROR - Stage preprocess failed for run 1: [json.exception.type_error.302] type must be number, but i

ConversionError: Conversion failed for: 하나생명(액티브)_251127.pdf with status: ConversionStatus.FAILURE. Errors: Page 1: [json.exception.type_error.302] type must be number, but is array; Page 2: [json.exception.type_error.302] type must be number, but is array; Page 3: [json.exception.type_error.302] type must be number, but is array; Page 4: [json.exception.type_error.302] type must be number, but is array

In [ ]:
# print("===========docling===========")
print(f"   {document_text}")

In [52]:
document_text = extract_text_from_pdf_with_pdfplumber(_document_file_path, _password)


In [53]:

# print("===========pdfplumber===========")
print(f"   {document_text}")

   | 화면번호 : | 13001 |
| --- | --- |
| 페이지 : | 1/4 |

| 위 탁 사 : |  |  |  |  |  |
| --- | --- | --- | --- | --- | --- |
| 수 탁 사 : | 신한은행(삼성액티브자산-하나생명변액) |  |  |  |  |
| 거래종류 : | 펀드설정(수탁은행용) |  |  |  | ( 단위 : 원 ) |

| 거래유형 |  | 펀드코드 |  | 사무관리사 | 결제일 | 설정(해지)전좌수 | 설정(해지)좌수 | 펀드납입(인출)금액 | 운용보수 | 수탁보수 | 펀드평가보수 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 판매사 |  | 펀드명 |  | 펀드평가사 | 통화코드 | 설정(해지)후좌수 | 설정(해지)금액 | 판매회사분결제액 | 판매보수 | 사무관리보수 | 성과보수 |
| 설정 |  | BBC13F |  | 하나펀드서비스 | 2025-11-27 | 18,577,453,950 | 3,406,206 | 7,108,103 |  |  |  |
| 하나생명 |  | VUL 주식성장형(1형)_SamsungActive |  |  |  | 18,580,860,156 | 7,108,103 | 7,108,103 |  |  |  |
|  |  | [매매처계] |  |  |  | 18,577,453,950 | 3,406,206 | 7,108,103 |  |  |  |
|  |  |  |  |  |  | 18,580,860,156 | 7,108,103 | 7,108,103 |  |  |  |
|  |  | [결제일계] |  |  |  | 18,577,453,950 | 3,406,206 | 7,108,103 |  |  |  |
|  |  |  |  |  |  | 18,580,860,156 | 7,108,103 | 7,108,103 |  |  |  |
| [수탁사계] |  |  |  |  |  | 18,577,453,9